# 03 · Train / validation / test split (without leakage)

**Problem:** near-identical seeds (e.g. the same sample for .NET 6 and .NET 7) can end up in both train and test.
The model would then be tested on code it has effectively already seen, and the score would be inflated.

**Fix:** cluster seeds that are ≥ 80% similar, then split by cluster: 80% train, 5% validation, 15% test.
Finally, format every example as a chat conversation (system / user / assistant) for fine-tuning.

## 1. Setup
Load the seeds (notebook 01) and the training examples (notebook 02).

In [1]:
# Cell 3: Setup.
import sys                                              # To add the project folder to Python's import path
from pathlib import Path                                # File paths that work on any OS

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # Project folder, wherever it lives
sys.path.insert(0, str(ROOT))                           # So our .py files can be imported

import random                                           # Shuffle clusters (seeded)
from difflib import SequenceMatcher                     # Measure how similar two programs are
import pandas as pd                                     # Tables

DATA = ROOT / "data" / "processed"                      # Where all processed data lives
seeds = pd.read_json(DATA / "seeds.jsonl", lines=True)  # Compiling programs (notebook 01)
ex = pd.read_json(DATA / "examples.jsonl", lines=True)  # Training examples (notebook 02)

SIMILARITY = 0.80                                       # Seeds at least this similar go in the same cluster
TEST_SHARE, VAL_SHARE = 0.15, 0.05                      # Share of examples for test and validation (rest = train)
EVAL_SIZE = 200                                         # Fixed test subset for before/after evaluation
SPLIT_SEED = 42                                         # Same split every time the notebook runs
print(f"{len(seeds)} seeds, {len(ex):,} examples")

367 seeds, 2,692 examples


## 2. Cluster near-duplicate seeds
Seeds from the same source file start in the same cluster; then any two seeds that are ≥ 80% similar are merged.
Uses a union-find structure: each cluster has one "root" seed that represents it.

In [2]:
# Cell 5: Merge near-duplicate seeds into clusters (takes about 1-3 minutes).
codes = seeds["code"].str.replace(r"\s+", " ", regex=True).tolist()   # Compare code ignoring spacing
n = len(codes)
parent = list(range(n))                                 # Union-find: at first every seed is its own cluster


def find(i):
    """The root (representative) of seed i's cluster."""
    while parent[i] != i:
        parent[i] = parent[parent[i]]                   # Shortcut the path (makes later lookups faster)
        i = parent[i]
    return i


def union(i, j):
    """Merge the clusters of seeds i and j."""
    parent[find(j)] = find(i)


first_in_group = {}                                     # 1) Seeds from the same source file / P3 task start together
for i, g in enumerate(seeds["group"]):
    if g in first_in_group:
        union(first_in_group[g], i)
    else:
        first_in_group[g] = i

merges = 0                                              # 2) Merge any two seeds that are very similar
for i in range(n):
    for j in range(i + 1, n):
        if find(i) == find(j):                          # Already in the same cluster
            continue
        sm = SequenceMatcher(None, codes[i], codes[j], autojunk=False)
        if sm.real_quick_ratio() >= SIMILARITY and sm.quick_ratio() >= SIMILARITY and sm.ratio() >= SIMILARITY:
            union(i, j)                                 # (the two quick checks skip most pairs cheaply)
            merges += 1

seeds["cluster"] = [f"c{find(i):04d}" for i in range(n)]   # Cluster name = its root seed's number
print(f"Groups: {seeds['group'].nunique()} -> clusters: {seeds['cluster'].nunique()} ({merges} similarity merges)")
print("Largest clusters (seeds):", seeds["cluster"].value_counts().head(5).tolist())

Groups: 127 -> clusters: 80 (47 similarity merges)
Largest clusters (seeds): [115, 76, 17, 14, 10]


## 3. Split by cluster
Clusters are shuffled (seeded) and assigned to test until it holds 15% of the examples, then validation until 5%, then train.
A whole cluster always goes to one side, so no near-copy of a test program is ever trained on.

In [3]:
# Cell 7: Assign each cluster to test, val or train, then check there is no overlap.
ex = ex.merge(seeds[["id", "cluster"]].rename(columns={"id": "seed_id"}), on="seed_id")   # Give each example its cluster
sizes = ex.groupby("cluster").size()                    # Number of examples per cluster
clusters = sorted(sizes.index)                          # Sorted first, so the shuffle is repeatable
random.Random(SPLIT_SEED).shuffle(clusters)             # Random order (same every run)

split_of, counts = {}, {"test": 0, "val": 0, "train": 0}
for c in clusters:                                      # Fill test first, then val, the rest is train
    if counts["test"] < TEST_SHARE * len(ex):
        side = "test"
    elif counts["val"] < VAL_SHARE * len(ex):
        side = "val"
    else:
        side = "train"
    split_of[c] = side
    counts[side] += sizes[c]
ex["split"] = ex["cluster"].map(split_of)               # Each example inherits its cluster's side

for side in ("train", "val", "test"):                   # Summary per side
    part = ex[ex["split"] == side]
    print(f"{side:<5}: {len(part):>5,} examples ({len(part) / len(ex):.0%}) | {part['cluster'].nunique():>3} clusters "
          f"| {part['seed_id'].nunique():>3} seeds")

overlap = set(ex.loc[ex.split == "train", "cluster"]) & set(ex.loc[ex.split == "test", "cluster"])
assert not overlap, "Leakage: a cluster is in both train and test!"   # Stops the notebook if the split is wrong
print("\nNo cluster appears in both train and test ✓")
first_kind = ex["kinds"].str.split("+").str[0]          # Main operator of each example
print("\nOperator mix per split (%):")
print((pd.crosstab(first_kind, ex["split"], normalize="columns") * 100).round(0).astype(int))

train: 2,146 examples (80%) |  51 clusters | 296 seeds
val  :   139 examples (5%) |   8 clusters |  16 seeds
test :   407 examples (15%) |  21 clusters |  55 seeds

No cluster appears in both train and test ✓

Operator mix per split (%):
split           test  train  val
kinds                           
drop_async         3      2    4
drop_closing      16     16   14
drop_new           5      4    6
drop_semicolon    16     15   12
remove_using       5      5   10
wrong_literal     10     10   14
wrong_local       15     16   14
wrong_member      15     16   13
wrong_type        14     16   12


## 4. Format as chat conversations and save
Each example becomes: **system** (the task), **user** (compiler errors + broken code), **assistant** (the fixed code).
The test file also gets a fixed random subset of 200 examples for the before/after evaluation.

In [4]:
# Cell 9: Turn every example into a chat conversation and save train/val/test files.
SYSTEM_PROMPT = ("You fix C# compiler errors in ASP.NET Core Program.cs files. "
                 "Return the complete corrected file in one ```csharp block. Change only what is needed to fix the errors.")


def to_messages(row):
    """The conversation the model learns from: task, then errors + broken code, then the fixed file."""
    user = f"Compiler errors:\n{row['errors']}\n\nCode:\n```csharp\n{row['broken']}\n```"   # What the model sees
    assistant = f"```csharp\n{row['fixed']}\n```"       # What it should answer
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]


ex["messages"] = ex.apply(to_messages, axis=1)          # Add the conversation to every row
keep = ["id", "seed_id", "cluster", "kinds", "error_codes", "n_errors", "broken", "fixed", "messages"]   # Columns to save

for side in ("train", "val", "test"):
    part = ex[ex["split"] == side][keep]
    part.to_json(DATA / f"{side}.jsonl", orient="records", lines=True, force_ascii=False)   # train.jsonl / val.jsonl / test.jsonl
    print(f"Saved {side}.jsonl ({len(part):,} examples)")

test_eval = ex[ex["split"] == "test"].sample(min(EVAL_SIZE, (ex.split == "test").sum()), random_state=SPLIT_SEED)[keep]
test_eval.to_json(DATA / "test_eval.jsonl", orient="records", lines=True, force_ascii=False)   # The fixed 200 for evaluation
print(f"Saved test_eval.jsonl ({len(test_eval)} examples)")

Saved train.jsonl (2,146 examples)
Saved val.jsonl (139 examples)
Saved test.jsonl (407 examples)
Saved test_eval.jsonl (200 examples)


## 5. One formatted example
Exactly what the model receives (system + user) and what it must learn to answer (assistant).

In [5]:
# Cell 11: Print one training conversation.
for m in ex[ex["split"] == "train"].iloc[0]["messages"]:   # The first training example
    print(f"===== {m['role'].upper()} =====")
    print(m["content"][:900], "\n")                     # Shortened, so the output stays readable

===== SYSTEM =====
You fix C# compiler errors in ASP.NET Core Program.cs files. Return the complete corrected file in one ```csharp block. Change only what is needed to fix the errors. 

===== USER =====
Compiler errors:
Line 9: error CS1002: ; expected
    > builder.Services.AddControllers()

Code:
```csharp
namespace APIWithControllers;

public class Program
{
    public static void Main(string[] args)
    {
        var builder = WebApplication.CreateBuilder(args);

        builder.Services.AddControllers()
        var app = builder.Build();

        app.UseHttpsRedirection();

        app.MapControllers();

        app.Run();
    }
}

``` 

===== ASSISTANT =====
```csharp
namespace APIWithControllers;

public class Program
{
    public static void Main(string[] args)
    {
        var builder = WebApplication.CreateBuilder(args);

        builder.Services.AddControllers();
        var app = builder.Build();

        app.UseHttpsRedirection();

        app.MapControllers();

        